# 購買予測の例

このノートブックでは、Excelデータを使用した購買予測の基本的な流れを示します。

## 1. ライブラリのインポート

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_excel_data, get_data_info
from src.preprocessor import preprocess_data, DataPreprocessor
from src.train import train_model, PurchasePredictionModel
from src.predict import predict, predict_with_confidence

%matplotlib inline
sns.set_style('whitegrid')

## 2. データの読み込み

In [ ]:
# Excelファイルからデータを読み込み
# df = load_excel_data('../data/raw/your_data.xlsx')

# サンプルデータの作成（実際のデータがない場合）
np.random.seed(42)
n_samples = 1000

df = pd.DataFrame({
    'age': np.random.randint(18, 70, n_samples),
    'income': np.random.randint(20000, 150000, n_samples),
    'visit_count': np.random.randint(1, 50, n_samples),
    'cart_items': np.random.randint(0, 20, n_samples),
    'browsing_time': np.random.randint(1, 120, n_samples),
    'gender': np.random.choice(['M', 'F'], n_samples),
    'member_type': np.random.choice(['bronze', 'silver', 'gold'], n_samples),
})

# 購買有無を生成（年齢、収入、訪問回数に基づく）
purchase_score = (
    df['income'] / 1000 +
    df['visit_count'] * 2 +
    df['cart_items'] * 5 +
    (df['member_type'] == 'gold').astype(int) * 50
)
df['purchased'] = (purchase_score > purchase_score.median()).astype(int)

print(f"データ形状: {df.shape}")
df.head()

## 3. データの確認

In [ ]:
get_data_info(df)

In [ ]:
# 購買率の確認
print("購買率:")
print(df['purchased'].value_counts(normalize=True))

# 可視化
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

df['age'].hist(bins=30, ax=axes[0, 0])
axes[0, 0].set_title('年齢分布')

df['income'].hist(bins=30, ax=axes[0, 1])
axes[0, 1].set_title('収入分布')

df.groupby('purchased')['visit_count'].mean().plot(kind='bar', ax=axes[1, 0])
axes[1, 0].set_title('購買有無別の訪問回数')

df['purchased'].value_counts().plot(kind='pie', autopct='%1.1f%%', ax=axes[1, 1])
axes[1, 1].set_title('購買率')

plt.tight_layout()
plt.show()

## 4. データの前処理

In [ ]:
# データの前処理
X_train, X_test, y_train, y_test = preprocess_data(
    df,
    target_column='purchased',
    categorical_columns=['gender', 'member_type'],
    test_size=0.2,
    random_state=42
)

print(f"\n学習データ形状: {X_train.shape}")
print(f"テストデータ形状: {X_test.shape}")

## 5. モデルの学習

In [ ]:
# Random Forestモデルの学習
rf_model = train_model(
    X_train, y_train,
    X_test, y_test,
    model_type='random_forest',
    n_estimators=100,
    max_depth=10,
    random_state=42
)

In [ ]:
# XGBoostモデルの学習
xgb_model = train_model(
    X_train, y_train,
    X_test, y_test,
    model_type='xgboost',
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

## 6. 予測の実行

In [ ]:
# テストデータで予測
predictions = rf_model.predict(X_test)
print(f"予測結果: {predictions[:10]}")
print(f"実際の値: {y_test[:10]}")

In [ ]:
# 信頼度付き予測
predictions_with_conf = predict_with_confidence(rf_model.model, X_test)
predictions_with_conf.head(10)

## 7. モデルの保存

In [ ]:
# モデルの保存
rf_model.save_model('../models/random_forest_model.pkl')
xgb_model.save_model('../models/xgboost_model.pkl')

## 8. 予測結果の可視化

In [ ]:
# 予測確率の分布
proba = rf_model.predict_proba(X_test)[:, 1]

plt.figure(figsize=(10, 6))
plt.hist([proba[y_test == 0], proba[y_test == 1]], bins=30, label=['購買なし', '購買あり'], alpha=0.7)
plt.xlabel('購買確率')
plt.ylabel('頻度')
plt.title('予測確率の分布')
plt.legend()
plt.show()